In [1]:
from mpramnist.Gosai2024.dataset import GosaiDataset

from mpramnist.Lee2025.dataset import LeeDataset
from mpramnist.Lee2025.trainer import LitModel_Lee

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import mpramnist.transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L
from lightning.pytorch.callbacks import ModelCheckpoint

from torchmetrics import PearsonCorrCoef

import pandas as pd

BATCH_SIZE = 1024
NUM_WORKERS = 8

I0000 00:00:1782551355.800180  948927 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782551356.649923  948927 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import warnings
warnings.filterwarnings("ignore")


# Lee SNP Dataset Description

Dataset contains results from Lee *et al* MPRA assay of **13,261 non-coding variants**, associated with eight psychiatric disorders, 
conducted in Human Neural Progenitor cells (HNPs). 

The assay used **150-bp** SNP-centered itervals for both reference and alternate alleles.


All variants were classified into 4 groups:
| **Variant class** | **Class label**| **Number of variants** | **Description** | 
| :-----------: | :-----------: | :-----------: | :-----------: |
| emVar | 1 | 683 | significant differential regulatory activity between risk and protective alleles (FDR<0.05) and location in MPRA-active element |
| MPRA-Allelic | 2 | 3789 | significant differential regulatory activity between risk and protective alleles (FDR<0.05) |
| MPRA-nonallelic | 3 | 6396 | no significant differential regulatory activity between risk and protective alleles (p>0.1) |
| Uncertain | 4 | 2393 | others |

# Current Workflow

In this notebook, we:

1. Train the **MPRALegNet** model on the **Gosai SK-N-SH dataset**

2. Assess its predictive power using **Lee's SNPs**

## **Train MPRALegNet model using Gosai SK-N-SH data**

In [ ]:
cell_types = ["SKNSH"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(0.5), t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

# load the data
train_dataset_own = GosaiDataset(
    split="train",
    transform=train_transform,
    filtration="own",
    cell_types=cell_types,
    stderr_columns=["SKNSH_lfcSE"], 
    stderr_threshold=1.0,
    std_multiple_cut=6.0,
    up_cutoff_move=3.0,
    duplication_cutoff=0.5,
    root="../data",
)
# Use the same parameters to valid and test
val_dataset_own = GosaiDataset(split="val", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='../data',)
test_dataset_own = GosaiDataset(split="test", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='../data',)

print(train_dataset_own)

Dataset GosaiDataset (MpraDaraset)
    Number of datapoints: 842024
    Root location: /media/storage/lizzzafomenko/data/Malinois
    Using split: ['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '14', '15', '16', '17', '18', '20', '22', 'Y']
    Split: {'train': 668946, 'val': 58809, 'test': 62582}
    Task: Regression
    Description: The Gosai dataset includes 798,064 sequences tested in the K562, HepG2, and SK-N-SH cell lines. The original sequence length is approximately 200 nucleotides, and it is recommended to extend them to 600 bp. The task is to predict three normalized regulatory activity values for the respective cell lines.


In [4]:
# encapsulate data into DataLoader form
train_loader = DataLoader(dataset=train_dataset_own, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset_own[0][0])
out_channels = len(cell_types)

In [ ]:
# initialize LegNet model

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Lee(
    model=model,
    loss=nn.MSELoss(),
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
    use_one_cycle = True
)

In [6]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_pearson", mode="max", save_top_k=1, save_last=False
)

trainer = L.Trainer(
    accelerator="gpu",
    devices=[3],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
    callbacks=[checkpoint_callback],
)

trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name          | Type            | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | tra

Sanity Checking: |          | 0/? [00:00<?, ?it/s]


----------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 1.64248 | Val Pearson: -0.05681 | Train Pearson: nan 
----------------------------------------------------------------------------



Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...


-------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 0.56968 | Val Pearson: 0.84908 | Train Pearson: 0.73036 
-------------------------------------------------------------------------------



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.4692828953266144
      test_pearson          0.7975703477859497
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.4692828953266144, 'test_pearson': 0.7975703477859497}]

In [ ]:
# load model from the best checkpoint

best_model_path = checkpoint_callback.best_model_path
seq_model = LitModel_Lee.load_from_checkpoint(best_model_path, model=model, loss=nn.MSELoss(), weight_decay=0.1, lr=0.01, print_each=1, use_one_cycle = True,)

## **Evaluate MPRALegNet model using Lee SNPs Data**

Initialize transformations for both `forward` and `reverse_complement` sequences.

For each sequence add flanks from Gosai assay and crop to the length of 600 base pairs.

In [3]:
forw_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])
revcomp_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(1), t.Seq2Tensor()])

In [ ]:
# initialize datasets and encapsulate data into DataLoader form

predict_forward_dataset = LeeDataset(split = 'test', length = 150, transform=forw_transform, root = '../data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = LeeDataset(split = 'test', length = 150,  transform=revcomp_transform, root = '../data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# get model predictions

forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

**emVar** - Variants with significant differential regulatory activity between risk and protective alleles (FDR<0.05) and location in MPRA-active elements

**daVar** - All variants with significant differential regulatory activity between risk and protective alleles (FDR<0.05)

**all** - All variants from the experiment

In [14]:
def Lee_variants_prediction(forw_preds, revcomp_preds, variant_types = ['emVar', 'daVar', 'all'], return_df = True):
    """
    Calculate Pearson correlation between model-predicted and MPRA-measured 
    variant effects for each cell type.

    Parameters
    ----------
    forw_preds : list of dict
        List of dictionaries containing model predictions for forward sequences.
        Each dict must have keys: 'target', 'fdr', 'ref_predicted', 'alt_predicted'
    revcomp_preds : list of dict
        List of dictionaries containing model predictions for reverse complement sequences.
        Same structure as forw_preds
    variant_types : list, optional
        Categories to evaluate: 'emVar', 'daVar' or 'all'.
        Default ['emVar', 'daVar', 'all'].
    return_df : bool
        If True, return results as pd.DataFrame; otherwise print them

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - variant_type: variant type
        - n: number of variants of this variant_type
        - pearsonr: pearson correlation score
    
    Note
    ----
    prediction_sign reverses effect (alt-ref to ref-alt) because MPRA variants logFC
    scores are calculated as log(effect GWAS / non-effect GWAS). 
    """

    MAP_VARIANTS_TYPE = {'emVar': [1], 'daVar': [1, 2], 'all': [1, 2, 3, 4]}

    targets = torch.cat([pred["target"] for pred in forw_preds])
    variant_type = torch.cat([pred["variant_type"] for pred in forw_preds])

    # prediction sign. some predictions should be reversed (ref-alt instead of alt-ref)
    # because dataset variant logFC is calculated as log(Major / Minor)
    prediction_sign = torch.cat([pred["reverse_prediction"] for pred in forw_preds])

    y_preds_forw_ref = torch.cat([pred["ref_predicted"] for pred in forw_preds])
    y_preds_forw_alt = torch.cat([pred["alt_predicted"] for pred in forw_preds])


    y_preds_revcomp_ref = torch.cat([pred["ref_predicted"] for pred in revcomp_preds])
    y_preds_revcomp_alt = torch.cat([pred["alt_predicted"] for pred in revcomp_preds])

    y_preds_ref = torch.mean(torch.stack([y_preds_forw_ref, y_preds_revcomp_ref]), dim=0)
    y_preds_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_revcomp_alt]), dim=0)

    variant_prediction = (y_preds_alt - y_preds_ref).squeeze() * prediction_sign.squeeze()
    
    results = []
    pears = PearsonCorrCoef()

    for i in range(len(variant_types)):

        mask = torch.isin(variant_type, torch.tensor(MAP_VARIANTS_TYPE[variant_types[i]]))
        pearsonr = pears(variant_prediction.squeeze()[mask], targets.squeeze()[mask])

        if return_df:
            results.append({
                'variant_type': variant_types[i],
                'n': mask.sum().item(),
                'pearsonr': pearsonr.item()
            })
        
        else:
             print(f'{variant_types[i]} (n = {mask.sum().item()}):   {pearsonr:.6f}')

    if return_df:
        df = pd.DataFrame(results)
        return df



In [ ]:
Lee_variants_prediction(forw_preds, revcomp_preds, ['emVar', 'daVar', 'all'], False)

emVar (n = 683):   0.013937
daVar (n = 4472):   0.007249
all (n = 13261):   0.003717


## Test AlphaGenome model

In [ ]:
from mpramnist.models import predict_variants_AlphaGenome, filter_tracks

I0000 00:00:1782315872.404262 3287004 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782315873.285835 3287004 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
# Use all available cell types
basic_transform = t.Compose([t.Seq2Tensor()])
ag_dataset = LeeDataset(split = 'test', length = 2**11, transform=basic_transform, root = '/media/storage/lizzzafomenko/data') 


In [5]:
ag_preds = predict_variants_AlphaGenome(weights_path = '/media/storage/lizzzafomenko/data/AlphaGenome/weights/model_all_folds.safetensors', 
                            dataset = ag_dataset, 
                            batch_size = 4, 
                            device = 'cuda:1') 


In [31]:
def Lee_AlphaGenome_variants_prediction(preds, variant_types = ['emVar', 'daVar', 'all'], return_df = True, biosample_names = None, exact_match = False):
    """
    Calculate Pearson correlation between model-predicted and MPRA-measured 
    variant effects for each cell type.

    Parameters
    ----------
    forw_preds : list of dict
        List of dictionaries containing model predictions for forward sequences.
        Each dict must have keys: 'target', 'fdr', 'ref_predicted', 'alt_predicted'
    revcomp_preds : list of dict
        List of dictionaries containing model predictions for reverse complement sequences.
        Same structure as forw_preds
    variant_types : list, optional
        Categories to evaluate: 'emVar', 'daVar' or 'all'.
        Default ['emVar', 'daVar', 'all'].
    biosample_names : list[list[str]]
        List of biosample names to be used to filter AlphaGenome tracks 
        before variant prediction. If None, all tracks will be used for prediction
    return_df : bool
        If True, return results as pd.DataFrame; otherwise print them

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - variant_type: variant type
        - n: number of variants of this variant_type
        - pearsonr: pearson correlation score
    
    Note
    ----
    prediction_sign reverses effect (alt-ref to ref-alt) because MPRA variants logFC
    scores are calculated as log(effect GWAS / non-effect GWAS). 
    """

    MAP_VARIANTS_TYPE = {'emVar': [1], 'daVar': [1, 2], 'all': [1, 2, 3, 4]}

    targets = torch.cat([pred["target"] for pred in preds])
    variant_type = torch.cat([pred["var_type"] for pred in preds])

    # prediction sign. some predictions should be reversed (ref-alt instead of alt-ref)
    # because dataset variant logFC is calculated as log(Major / Minor)
    prediction_sign = torch.cat([pred["reverse_prediction"] for pred in preds])

    # sequence prediction
    y_preds_ref = torch.cat([pred["ref_predicted"] for pred in preds])
    y_preds_alt = torch.cat([pred["alt_predicted"] for pred in preds])

    variant_prediction = y_preds_alt - y_preds_ref
    
    results = []
    pears = PearsonCorrCoef()

    for i in range(len(variant_types)):

        if biosample_names:
            var_preds = filter_tracks(variant_prediction, biosample_names, exact_match = exact_match)
            var_preds = var_preds.mean(axis = 1)
        else:
            var_preds = variant_prediction.mean(axis = 1)

        var_preds = var_preds * prediction_sign

        mask = torch.isin(variant_type, torch.tensor(MAP_VARIANTS_TYPE[variant_types[i]]))
        pearsonr = pears(var_preds.squeeze()[mask], targets.squeeze()[mask])

        if return_df:
            results.append({
                'variant_type': variant_types[i],
                'n': mask.sum().item(),
                'pearsonr': pearsonr.item()
            })
        
        else:
             print(f'{variant_types[i]} (n = {mask.sum().item()}):   {pearsonr:.6f}')

    if return_df:
        df = pd.DataFrame(results)
        return df

In [32]:
# specify biosample for AlphaGenome predictions

Lee_AlphaGenome_variants_prediction(ag_preds, ['emVar', 'daVar', 'all'], True, ['neuronal stem cell'], True)

,variant_type,n,pearsonr
0,emVar,683,0.033157
1,daVar,4472,-0.001590
2,all,13261,-0.005772


In [33]:
Lee_AlphaGenome_variants_prediction(ag_preds, ['emVar', 'daVar', 'all'], True)

,variant_type,n,pearsonr
0,emVar,683,0.013011
1,daVar,4472,0.000346
2,all,13261,0.001853


# DELETE ME

In [9]:
from mpramnist.models import BassetBranched, PARM, DREAM_RNN

In [6]:
# initialize datasets and encapsulate data into DataLoader form

predict_forward_dataset = LeeDataset(split = 'test', length = 150, transform=forw_transform, root = '/media/storage/lizzzafomenko/data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = LeeDataset(split = 'test', length = 150,  transform=revcomp_transform, root = '/media/storage/lizzzafomenko/data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [8]:
trainer = L.Trainer(
    accelerator="cpu",
    #devices=[3],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [10]:
model = DREAM_RNN(in_channels=len(predict_forward_dataset[0][0]['seq']), seqsize=600, out_channels=1)

seq_model = LitModel_Lee(
    model=model,
    loss=nn.MSELoss(),
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
    use_one_cycle = True
)

In [11]:
# DREAM-RNN

chekpoints = [
'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_0/checkpoints/epoch=42-step=35389.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_1/checkpoints/epoch=47-step=39504.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_2/checkpoints/epoch=0-step=823.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_3/checkpoints/epoch=42-step=35389.ckpt',

'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Gosai2024/lightning_logs/version_4/checkpoints/epoch=47-step=39504.ckpt',
]

In [15]:
for i, ckp in enumerate(chekpoints):
    seq_model = LitModel_Lee.load_from_checkpoint(ckp, model=model, loss=nn.MSELoss(), weight_decay=0.1, lr=0.01, print_each=1, use_one_cycle = True,)
    forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
    revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)
    df = Lee_variants_prediction(forw_preds, revcomp_preds, ['emVar', 'daVar', 'all'], True)
    df.to_csv(f'/mnt/calc/lizzzafomenko/MPRA-MNIST/mpramnist/Lee2025/results/DREAM_RNN_run{i}.tsv', sep='\t', index = False)

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()

Output()